# NCSound engine: e2e + component benchmark

Runs the JUCE-free half of the project on Kaggle compute: the full engine test
suite (correctness) and `nc_bench` (per-kernel timings), then compares every
kernel against the stored baseline and saves the run as an artifact.

**Enable Internet** in the notebook settings: the build fetches Catch2 through
CPM. Set `REPO_URL` below to this project's git remote.


In [ ]:
import json, os, pathlib, subprocess

REPO_URL = os.environ.get("REPO_URL", "")   # e.g. https://github.com/<you>/<repo>.git
WORK = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").exists() else pathlib.Path.cwd()
REPO = WORK / "repo"

if not REPO.exists():
    if not REPO_URL:
        raise SystemExit("Set REPO_URL to the git remote, or upload the repo to /kaggle/working/repo")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

print(REPO)


In [ ]:
# Kaggle images ship cmake and g++ but not always ninja.
!apt-get -qq update > /dev/null 2>&1
!apt-get -qq install -y ninja-build > /dev/null 2>&1
!cmake --version | head -1; g++ --version | head -1; nproc


In [ ]:
# Configure, build nc_bench + nc_tests, run the engine suite, write the JSON.
# The suite gates the numbers: a failing assertion fails this cell.
log = WORK / "build.log"
env = dict(os.environ, OUT=str(WORK / "run.json"))
with open(log, "w") as fh:
    proc = subprocess.run(["bash", "tools/kaggle/run_bench.sh"], cwd=REPO,
                          env=env, stdout=fh, stderr=subprocess.STDOUT)
print("exit", proc.returncode)
print("".join(log.read_text(errors="replace").splitlines(keepends=True)[-25:]))
assert proc.returncode == 0, "build/test/bench failed - see build.log"


In [ ]:
import pandas as pd

run = json.loads((WORK / "run.json").read_text())
df = pd.DataFrame(run["kernels"])[["name", "units", "ops", "ns_per_op"]]
print(run["run"])
df.sort_values("ns_per_op", ascending=False)


In [ ]:
# Compare against the committed baseline. Same-machine-class comparison only:
# the baseline is one dev box, so treat the ratio as a hint about direction.
baseline_path = REPO / "tools/kaggle/baseline.json"
if baseline_path.exists():
    base = {k["name"]: k["ns_per_op"] for k in json.loads(baseline_path.read_text())["kernels"]}
    df["baseline_ns"] = df["name"].map(base)
    df["ratio"] = df["ns_per_op"] / df["baseline_ns"]
    flagged = df[df["ratio"] > 1.25].sort_values("ratio", ascending=False)
    print(f"{len(flagged)} kernel(s) above 1.25x baseline")
    display(flagged[["name", "ns_per_op", "baseline_ns", "ratio"]])
else:
    print("no baseline in the clone; this run becomes the reference")


In [ ]:
# Artifacts: CSV for tracking across runs, plus a bar chart of ns/op.
out = WORK / "nc_bench.csv"
df.drop(columns=[c for c in ("baseline_ns", "ratio") if c in df], errors="ignore").to_csv(out, index=False)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(df["name"], df["ns_per_op"])
ax.set_xscale("log")
ax.set_xlabel("ns per op (log)")
ax.set_title("nc_bench: component cost")
plt.tight_layout()
fig.savefig(WORK / "nc_bench.png", dpi=120)
print("wrote", out)


## Limits worth remembering

- Kaggle has no audio device and no JUCE build target, so nothing here measures
  the real-time block path (`renderBlock` and its passes). Those need the
  Windows bench target, which drives `processBlockForTesting` without hardware.
- Absolute `ns_per_op` is machine-specific. Compare the same kernel between two
  runs of the same machine class; the ratio column is a question, not a verdict.
- `resample.bank_build` and `export.measure_loudness_10s` are message-thread and
  offline paths. `spsc_queue.push_pop`, `resample.weights_lookup` and
  `tempo_map.*` are on the audio path and are the ones with a hard budget.

## Next

1. Add the Windows `nc_bench_render` target so the block path is tracked the
   same way (per block, at 64/256/1024 frames, with and without inserts).
2. Once two clean runs exist per component, start optimisation from the kernels
   with the worst ratio, not from the ones that look big in the source.
